In [1]:
import streamlit as st
import cv2
import numpy as np
import joblib
import os
from skimage.feature import hog
from PIL import Image

# ---------------- PAGE CONFIG ----------------
st.set_page_config(
    page_title="FaceGuard AI | Mask Detection",
    page_icon="😷",
    layout="wide"
)

# ---------------- CUSTOM DESIGN ----------------
st.markdown("""
<style>
.stApp {
    background: linear-gradient(135deg, #0b1220, #152943);
    color: white;
}
.main-title {
    font-size: 42px;
    font-weight: 800;
    color: #65d6ff;
}
.subtitle {
    color: #c4d4e5;
    font-size: 17px;
}
.metric-card {
    background: #1d314b;
    padding: 20px;
    border-radius: 14px;
    border: 1px solid #35516e;
}
</style>
""", unsafe_allow_html=True)

# ---------------- LOAD MODEL ----------------
MODEL_PATH = os.path.join("models", "mask_svm.pkl")
CASCADE_PATH = os.path.join(
    os.getcwd(), "haarcascade_frontalface_default.xml"
)

@st.cache_resource
def load_resources():
    model = joblib.load(MODEL_PATH)
    detector = cv2.CascadeClassifier(CASCADE_PATH)

    if detector.empty():
        raise RuntimeError("Face detector file load nahi hua.")

    return model, detector

st.markdown(
    '<div class="main-title">FaceGuard AI</div>',
    unsafe_allow_html=True
)
st.markdown(
    '<div class="subtitle">Traditional Machine Learning • OpenCV • HOG + SVM</div>',
    unsafe_allow_html=True
)
st.divider()

if not os.path.exists(MODEL_PATH):
    st.error("Model file nahi mili. models/mask_svm.pkl check karo.")
    st.stop()

if not os.path.exists(CASCADE_PATH):
    st.error("haarcascade_frontalface_default.xml app.py ke folder mein rakho.")
    st.stop()

try:
    model, face_detector = load_resources()
except Exception as e:
    st.error(f"Resources load nahi hue: {e}")
    st.stop()

# ---------------- SIDEBAR ----------------
st.sidebar.title("FaceGuard AI")
st.sidebar.write("Mask Detection Dashboard")
st.sidebar.info(
    "Upload a clear face image. The system detects faces "
    "and classifies them using a trained SVM model."
)
st.sidebar.caption("Model: HOG + Linear SVM")

# ---------------- UPLOAD ----------------
st.subheader("Upload an Image")
uploaded_file = st.file_uploader(
    "Choose a JPG, JPEG or PNG image",
    type=["jpg", "jpeg", "png"]
)

if uploaded_file is not None:
    pil_image = Image.open(uploaded_file).convert("RGB")
    image_rgb = np.array(pil_image)
    image_bgr = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2BGR)

    gray_full = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)

    faces = face_detector.detectMultiScale(
        gray_full,
        scaleFactor=1.1,
        minNeighbors=5,
        minSize=(40, 40)
    )

    result_image = image_rgb.copy()
    mask_count = 0
    no_mask_count = 0

    for (x, y, w, h) in faces:
        face_gray = gray_full[y:y+h, x:x+w]
        face_gray = cv2.resize(face_gray, (64, 64))

        features = hog(
            face_gray,
            orientations=9,
            pixels_per_cell=(8, 8),
            cells_per_block=(2, 2),
            block_norm="L2-Hys"
        ).reshape(1, -1)

        prediction = int(model.predict(features)[0])

        # Labels: 1 = with mask, 0 = without mask
        if prediction == 1:
            label = "MASK"
            box_color = (30, 200, 100)
            mask_count += 1
        else:
            label = "NO MASK"
            box_color = (240, 70, 70)
            no_mask_count += 1

        cv2.rectangle(
            result_image, (x, y), (x+w, y+h), box_color, 3
        )
        cv2.putText(
            result_image,
            label,
            (x, max(y-10, 25)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.75,
            box_color,
            2
        )

    col1, col2 = st.columns(2)

    with col1:
        st.subheader("Original Image")
        st.image(image_rgb, use_container_width=True)

    with col2:
        st.subheader("Detection Result")
        st.image(result_image, use_container_width=True)

    st.divider()
    st.subheader("Detection Summary")

    m1, m2, m3 = st.columns(3)
    m1.metric("Faces Detected", len(faces))
    m2.metric("With Mask", mask_count)
    m3.metric("Without Mask", no_mask_count)

    if len(faces) == 0:
        st.warning(
            "Koi face detect nahi hua. Please clear, front-facing image try karo."
        )
    else:
        st.success("Image processing complete.")

    st.caption(
        "Note: Predictions are model estimates, not guaranteed results. "
        "Performance may vary with lighting, face angle and image quality."
    )

else:
    st.info("Start by uploading an image above.")
    st.markdown("""
    **How it works**
    1. Upload a face image.
    2. OpenCV detects faces.
    3. HOG extracts image features.
    4. The trained SVM classifies each detected face.
    """)

2026-10-29 00:25:20.381 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-29 00:25:20.384 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-29 00:25:21.464 
  command:

    streamlit run C:\Users\Nandani\anaconda3\anaconda\Lib\site-packages\ipykernel_launcher.py [ARGUMENTS]
2026-10-29 00:25:21.464 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-29 00:25:21.465 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-29 00:25:21.471 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-29 00:25:21.472 Thread 'MainThread': missing ScriptRunContext! This warning can be ignore

In [4]:
%%writefile app.py
import streamlit as st
import cv2
import numpy as np
import joblib
import os
from PIL import Image
from skimage.feature import hog

st.set_page_config(
    page_title="FaceGuard AI",
    page_icon="😷",
    layout="wide"
)

st.markdown("""
<style>
.stApp {
    background: linear-gradient(135deg, #0b1220, #172d49);
    color: white;
}
h1 { color: #62d5ff; }
</style>
""", unsafe_allow_html=True)

st.title("😷 FaceGuard AI")
st.write("Face Mask Detection using OpenCV + HOG + SVM")
st.caption("Traditional Machine Learning Project")
st.divider()

MODEL_PATH = "models/mask_svm.pkl"
CASCADE_PATH = "haarcascade_frontalface_default.xml"

if not os.path.exists(MODEL_PATH):
    st.error("Model file nahi mili: models/mask_svm.pkl")
    st.stop()

if not os.path.exists(CASCADE_PATH):
    st.error("Haar Cascade XML file nahi mili.")
    st.stop()

@st.cache_resource
def load_model_and_detector():
    model = joblib.load(MODEL_PATH)
    detector = cv2.CascadeClassifier(CASCADE_PATH)
    if detector.empty():
        raise ValueError("Face detector load nahi hua.")
    return model, detector

try:
    model, detector = load_model_and_detector()
except Exception as e:
    st.error(f"Loading error: {e}")
    st.stop()

st.sidebar.header("About this project")
st.sidebar.write("1. Upload a face image")
st.sidebar.write("2. OpenCV detects faces")
st.sidebar.write("3. HOG extracts features")
st.sidebar.write("4. SVM predicts the class")

uploaded = st.file_uploader(
    "Upload a clear face image",
    type=["jpg", "jpeg", "png"]
)

if uploaded:
    pil_image = Image.open(uploaded).convert("RGB")
    rgb = np.array(pil_image)
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)

    faces = detector.detectMultiScale(
        gray,
        scaleFactor=1.1,
        minNeighbors=5,
        minSize=(40, 40)
    )

    result = rgb.copy()
    mask_count = 0
    no_mask_count = 0

    for (x, y, w, h) in faces:
        face = gray[y:y+h, x:x+w]
        face = cv2.resize(face, (64, 64))

        features = hog(
            face,
            orientations=9,
            pixels_per_cell=(8, 8),
            cells_per_block=(2, 2),
            block_norm="L2-Hys"
        ).reshape(1, -1)

        prediction = int(model.predict(features)[0])

        if prediction == 1:
            label = "WITH MASK"
            color = (40, 220, 100)
            mask_count += 1
        else:
            label = "WITHOUT MASK"
            color = (255, 70, 70)
            no_mask_count += 1

        cv2.rectangle(result, (x, y), (x+w, y+h), color, 3)
        cv2.putText(
            result, label, (x, max(y-10, 25)),
            cv2.FONT_HERSHEY_SIMPLEX, 0.65, color, 2
        )

    left, right = st.columns(2)

    with left:
        st.subheader("Original Image")
        st.image(rgb, use_container_width=True)

    with right:
        st.subheader("Detection Result")
        st.image(result, use_container_width=True)

    st.divider()
    a, b, c = st.columns(3)
    a.metric("Faces Detected", len(faces))
    b.metric("With Mask", mask_count)
    c.metric("Without Mask", no_mask_count)

    if len(faces) == 0:
        st.warning("Face detect nahi hua. Clear, front-facing photo try karo.")
    else:
        st.success("Image processing complete!")

    st.info(
        "Note: Predictions are estimates. Model performance may vary "
        "with image quality, face angle and lighting."
    )
else:
    st.info("Upload an image to start detection.")

Overwriting app.py


In [5]:
import os

print("Project folder:", os.getcwd())
print("app.py exists:", os.path.exists("app.py"))
print("Model exists:", os.path.exists("models/mask_svm.pkl"))

Project folder: C:\Users\Nandani\face_mask_detection
app.py exists: True
Model exists: True
